# 00 · Hello Pegasus: is everything working?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vorhersager/gw1b-f2026/blob/main/notebooks/00_hello_pegasus.ipynb)

*Colab:* after the notebook opens, **Connect ▾ → Connect to a local runtime** and paste the URL printed by `gw1b jupyter` on Pegasus (one-step from the laptop: `laptop/gw1b-connect.sh <netid>`; details in [docs/COLAB.md](https://github.com/vorhersager/gw1b-f2026/blob/main/docs/COLAB.md)). The code then runs on a Pegasus GPU node. *JupyterLab / Open OnDemand:* just run the cells.

Welcome to the GW1B environment. This notebook checks that everything works and shows the
few things you need to know about where files live and how much compute you have.

**How you got here:** on Pegasus you ran `gw1b jupyter`, opened the ssh tunnel on your laptop, and
opened `http://localhost:8888/lab?token=...` — or connected Google Colab to it via *Connect → Connect to a local runtime*.
Either way, **the code runs on a Pegasus GPU node**, not on your laptop or on Google's servers.

In [ ]:
# --- GW1B setup cell (same in every notebook) -------------------------------------------------
import os, sys, json, time, pathlib
GW1B_HOME    = os.environ.get("GW1B_HOME", "/scratch/gw1b-class/group/gw1b-f2026")
GW1B_SCRATCH = os.environ.get("GW1B_SCRATCH", "/scratch/gw1b-class")
GW1B_GROUP   = os.environ.get("GW1B_GROUP", "/scratch/gw1b-class/group")
if not os.path.isdir(GW1B_HOME):
    raise RuntimeError(f"This kernel is not running on Pegasus ({GW1B_HOME} does not exist). In Colab use "
                       "Connect ▾ → 'Connect to a local runtime' and paste the URL printed by `gw1b jupyter` "
                       "(docs/COLAB.md); a Google-hosted runtime does not have the class environment.")
USER_DIR     = os.path.join(GW1B_SCRATCH, "users", os.environ.get("USER", "student"))
os.makedirs(USER_DIR, exist_ok=True)
if GW1B_HOME not in sys.path:
    sys.path.insert(0, GW1B_HOME)        # the shared gw1b package
import jax, jax.numpy as jnp, numpy as np
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print("your scratch dir:", USER_DIR)


## 1. Do we have a GPU, and how fast is it?

In [ ]:
dev = jax.devices()[0]
print("device kind:", dev.device_kind, "| platform:", dev.platform)
x = jnp.ones((4096, 4096), jnp.bfloat16 if dev.platform == "gpu" else jnp.float32)
f = jax.jit(lambda a: (a @ a).sum())
f(x).block_until_ready()                      # first call compiles
t = time.time(); [f(x).block_until_ready() for _ in range(5)]; dt = (time.time() - t) / 5
print(f"matmul throughput: {2 * 4096**3 / dt / 1e12:.1f} TFLOP/s  (A100 bf16 peak ~312, V100 ~15 in f32)")

## 2. Where things live

| what | where | notes |
|---|---|---|
| shared environment, code, tokenizer | `$GW1B_GROUP` (currently `/scratch/gw1b-class/group`) | persistent, read-only for students |
| datasets, HF cache, checkpoints | `$GW1B_SCRATCH` (`/scratch/gw1b-class`) | fast GPFS, **not backed up** — copy results out |
| your experiments | `$GW1B_SCRATCH/users/<netid>/runs/<run name>` | `config.yaml`, `metrics.jsonl`, `checkpoints/`, `summary.json` |
| your home | `~` (25 GB quota) | notebooks, small files only |

In [ ]:
for p in [GW1B_HOME, GW1B_GROUP, GW1B_SCRATCH, USER_DIR]:
    print(f"{'exists' if os.path.exists(p) else 'MISSING':8s} {p}")
data_dir = os.path.join(GW1B_SCRATCH, "data")
print("\ndatasets available:", os.listdir(data_dir) if os.path.isdir(data_dir) else "(none yet)")

## 3. Compute is the experimental variable

Every GW1B experiment reports *performance, tokens, FLOPs, GPU-hours and dollars*. The budget calculator
tells you what a run costs **before** you submit it.

In [ ]:
from gw1b.budget import estimate, format_estimate
from gw1b.config import load_config
for cfg_name in ["50m", "350m", "gw1b_1p15b"]:
    cfg = load_config(os.path.join(GW1B_HOME, "configs", cfg_name + ".yaml"))
    tokens = 20 * cfg.model.n_params                         # Chinchilla-optimal
    e = estimate(cfg.model.n_params, tokens, gpu="a100", n_gpus=8, mfu=0.35,
                 n_layers=cfg.model.n_layers, seq_len=cfg.data.seq_len, d_model=cfg.model.d_model)
    print(cfg_name, "->", format_estimate(e).splitlines()[1].strip())

## 4. The workflow in one picture

```
laptop (browser / Colab UI) ──ssh tunnel──▶ Pegasus login node ──Slurm──▶ GPU node: JupyterLab + your kernel
                                                                    └──▶ batch jobs: gw1b run -m gw1b.train ...
```

* Notebooks are for **exploring**: tokenizers, data, small models, plots, checkpoints.
* Anything longer than ~1 hour goes to a **batch job** (`gw1b run` / `gw1b train`) so it survives your laptop going to sleep,
  and resumes automatically from its last checkpoint if it hits the time limit (`--chain N`).

Next: `01_tokenizer.ipynb`.